# Week 6 — Day 2: Data Pre-processing

Use an LLM to rewrite each product into a standard, concise format. This drastically reduces noise and gives downstream models a much cleaner signal.

Approach: send 1,000-item batches through Groq's batch API (using `gpt-oss-20b` with low reasoning effort) — much cheaper than synchronous calls.

**Cost estimate:** under $1 for the lite (20K) dataset, ~$30 for the full (800K) dataset. You can also skip pre-processing entirely and load the already-processed dataset from HuggingFace in day 3.

## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from litellm import completion
from groq import Groq
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

## Pick lite or full mode

- `LITE_MODE = True` → 20,000 training items, costs under $1
- `LITE_MODE = False` → 800,000 training items, costs ~$30

In [ ]:
LITE_MODE = True

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
# Give every item a stable id so we can map batch results back to items
for index, item in enumerate(items):
    item.id = index

## The rewriting prompt

Asks the model for a fixed-format summary: title, category, brand, description, details.

In [ ]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

## Sanity-check with single (synchronous) calls

Try Groq's hosted gpt-oss-20b first, then a local Ollama model for comparison.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

## Walk through the batch API end-to-end on a 1,000-item sample

Each line of the JSONL file is one chat-completion request. Groq processes them asynchronously and returns one JSONL line of results per input.

In [ ]:
MODEL = "openai/gpt-oss-20b"

In [ ]:
def make_jsonl(item):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full}
        ],
        "reasoning_effort": "low"
    }
    line = {
        "custom_id": str(item.id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": body
    }
    return json.dumps(line)

In [ ]:
make_jsonl(items[0])

In [ ]:
def make_file(start, end, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=file_id
)
response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary

In [ ]:
print(items[0].full)
print()
print("---")
print()
print(items[0].summary)

## Use the Batch class to scale to the full dataset

`Batch` divides items into groups of 1,000, kicks off all batches, and lets us monitor and collect results when complete.

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
# Spot-check for any items that didn't get a summary
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

## Cleanup and push the processed dataset to the Hub

In [ ]:
# Drop fields we don't need on the Hub
for item in items:
    item.full = None
    item.id = None

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)